# Logistic regression of $Y^{\text{time}}$

**Purpose:** predict which merchants not yet using Subscriptions start using it within $H$ days, using only data from before the snapshot date.

- **Input:** `CLEAN_DATA_PATH` (clean merchant-day data), `COVARIATES_PATH` (static classes from `covariates.ipynb`)
- **Output:** `TARGET_LIST_PATH` (ranked target list)

**Steps**
1. Choose the snapshot dates $t_0$, the horizon $H$ and the minimum history $L$.
2. Build the sample at a snapshot date: risk set, outcome, covariates.
3. Fit on monthly rolling training snapshots, stacked.
4. Evaluate on a later test snapshot.
5. Score merchants at the end of the data and build the target list.

**Outcome:** for merchant $i$ with first payment $F_i$ and first Subscriptions payment $A_i$ ($A_i = \infty$ if never),

$$Y^{\text{time}}_i = \mathbf{1}\{t_0 \le A_i < t_0 + H\}, \qquad i \in \mathcal{R}(t_0) = \{i : F_i \le t_0 - L, \; A_i \ge t_0, \; \text{payment in } [t_0 - W, t_0)\}.$$

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import average_precision_score, roc_auc_score

CLEAN_DATA_PATH = Path("output/merchant_day.pkl")
COVARIATES_PATH = Path("output/merchant_covariates.pkl")
TARGET_LIST_PATH = Path("output/target_list_time.csv")  # logistic-only list; the combined list is built in rf.ipynb

W = 90           # covariate window: days before t0
TOP_K = 100      # list size

In [2]:
df = pd.read_pickle(CLEAN_DATA_PATH)
covariates = pd.read_pickle(COVARIATES_PATH)
static = covariates[["industry", "country", "business_size"]]
START, END = df["date"].min(), df["date"].max()

first_seen = df.groupby("merchant")["date"].min()
first_subscription = df.loc[df["subscription_volume"] > 0].groupby("merchant")["date"].min()
# Tenure clock: first charge date, or first day seen if that is earlier or missing
first_charge = covariates["first_charge_date"].combine(first_seen.reindex(covariates.index), lambda a, b: b if pd.isna(a) else min(a, b))
print(f"data: {START.date()} to {END.date()}, {(END - START).days + 1} days in total")

data: 2041-05-01 to 2042-06-22, 418 days in total


## 1. Choose $t_0$, $H$ and $L$

- $t_0$ (snapshot date): covariates come from $[t_0 - W, t_0)$, the outcome from $[t_0, t_0 + H)$.
- $H$ (horizon): days after $t_0$ in which adoption counts.
- $L$ (minimum history): days a merchant must have been observed before $t_0$.

**Design:** the test snapshot is the latest possible, $T + 1 - H$, so its outcome window ends on the last day of data. Training uses monthly rolling snapshots (the 1st of each month) with at least $\max(L, W)$ days of history and an outcome window that ends before the test snapshot.

### Sample size for each $(L, H)$

For each $(L, H)$: how many training snapshots fit, the stacked training rows and positives, how many distinct merchants those positives are (a merchant can be positive in several overlapping snapshots), and the test sample.

In [3]:
L_GRID = [0, 30, 60, 90]
H_GRID = [30, 60, 90, 120]


def snapshot_counts(t0, L_, H_):
    """Risk set at t0 for minimum history L_; returns its merchants and those with Y_time = 1 within H_."""
    active = df.loc[(df["date"] >= t0 - pd.Timedelta(days=W)) & (df["date"] < t0), "merchant"].unique()
    observed = (t0 - first_seen.reindex(active)).dt.days
    start = first_subscription.reindex(active)
    eligible = (observed >= L_) & ~(start < t0)
    y = (start >= t0) & (start < t0 + pd.Timedelta(days=H_))
    return eligible[eligible].index, y[eligible & y].index


rows = []
for H_ in H_GRID:
    test_t0 = END + pd.Timedelta(days=1) - pd.Timedelta(days=H_)
    for L_ in L_GRID:
        train_t0s = [
            t0 for t0 in pd.date_range(START, END, freq="MS")
            if (t0 - START).days >= max(L_, W) and t0 + pd.Timedelta(days=H_) <= test_t0
        ]
        train = [snapshot_counts(t0, L_, H_) for t0 in train_t0s]
        test_merchants, test_adopters = snapshot_counts(test_t0, L_, H_)
        train_rows = sum(len(m) for m, _ in train)
        train_pos = sum(len(a) for _, a in train)
        rows.append({
            "H": H_, "L": L_,
            "training snapshots": len(train_t0s),
            "training rows": train_rows,
            "training Y_time = 1": train_pos,
            "distinct adopters in training": len(set().union(*[set(a) for _, a in train])),
            "training rate (%)": round(train_pos / train_rows * 100, 2),
            "test snapshot": test_t0.date(),
            "test merchants": len(test_merchants),
            "test Y_time = 1": len(test_adopters),
        })
grid = pd.DataFrame(rows)
grid

,H,L,training snapshots,training rows,training Y_time = 1,distinct adopters in training,training rate (%),test snapshot,test merchants,test Y_time = 1
0,30,0,9,102886,491,485,0.48,2042-05-24,11993,65
1,30,30,9,95262,368,364,0.39,2042-05-24,11306,53
2,30,60,9,86692,306,302,0.35,2042-05-24,10618,47
3,30,90,9,74562,257,253,0.34,2042-05-24,9934,40
4,60,0,7,79373,649,420,0.82,2042-04-24,11928,115
5,60,30,7,73157,518,325,0.71,2042-04-24,11234,94
6,60,60,7,66031,436,272,0.66,2042-04-24,10540,77
7,60,90,7,55384,342,215,0.62,2042-04-24,9781,71
8,90,0,5,56057,645,358,1.15,2042-03-25,11689,142
9,90,30,5,51310,520,279,1.01,2042-03-25,11005,110


### Chosen values

In [4]:
H = 90
L = 30
TEST_T0 = END + pd.Timedelta(days=1) - pd.Timedelta(days=H)  # latest test snapshot: outcome window ends on the last day of data

# Monthly rolling training snapshots: enough history before t0, outcome window ends before TEST_T0
TRAIN_SNAPSHOTS = [
    t0 for t0 in pd.date_range(START, END, freq="MS")
    if (t0 - START).days >= max(L, W) and t0 + pd.Timedelta(days=H) <= TEST_T0
]
assert TRAIN_SNAPSHOTS, "no training snapshot fits before the test snapshot"
assert TEST_T0 + pd.Timedelta(days=H) <= END + pd.Timedelta(days=1), "test outcome window goes past the data"

print("training snapshots:", [t0.date().isoformat() for t0 in TRAIN_SNAPSHOTS])
print(f"test snapshot: {TEST_T0.date()}, {(TEST_T0 - START).days} days before it, {(END - TEST_T0).days + 1} days from it to the end of data")

training snapshots: ['2041-08-01', '2041-09-01', '2041-10-01', '2041-11-01', '2041-12-01']
test snapshot: 2042-03-25, 328 days before it, 90 days from it to the end of data


## 2. Sample at a snapshot date

Covariates use only payments in $[t_0 - W, t_0)$; merchants in the risk set have no Subscriptions volume before $t_0$, so `total_volume` is non-Subscriptions volume.

- `log_volume`, `log_checkout_volume`, `log_payment_link_volume`: $\log(1 + \text{volume in the window})$
- `active_rate`: days with volume in the window ÷ $\min(W, \text{days observed})$
- `recency`: days from the last payment to $t_0 - 1$
- `log_tenure`: $\log(t_0 - \text{first charge date})$ in days; tenure is right-skewed, and adoption falls roughly linearly in log tenure
- `industry` [Others], `country` [US], `business_size` [small]: classes from `covariates.ipynb`

In [5]:
REFERENCE = {"industry": "Others", "country": "US", "business_size": "small"}
CONTINUOUS = ["log_volume", "log_checkout_volume", "log_payment_link_volume", "active_rate", "recency", "log_tenure"]


def build_sample(t0, with_outcome=True):
    window = df[(df["date"] >= t0 - pd.Timedelta(days=W)) & (df["date"] < t0)]
    g = window.groupby("merchant")
    start = first_subscription.reindex(g.size().index)
    observed = (t0 - first_seen.reindex(g.size().index)).dt.days
    risk_set = (observed >= L) & ~(start < t0)

    s = static.loc[risk_set[risk_set].index].copy()
    g = window[window["merchant"].isin(s.index)].groupby("merchant")
    s["log_volume"] = np.log1p(g["total_volume"].sum())
    s["log_checkout_volume"] = np.log1p(g["checkout_volume"].sum())
    s["log_payment_link_volume"] = np.log1p(g["payment_link_volume"].sum())
    s["active_rate"] = g.size() / np.minimum(W, observed.reindex(s.index))
    s["recency"] = (t0 - pd.Timedelta(days=1) - g["date"].max()).dt.days
    s["log_tenure"] = np.log((t0 - first_charge.reindex(s.index)).dt.days)  # tenure >= L days, so no +1 needed
    if with_outcome:
        s["y"] = ((start >= t0) & (start < t0 + pd.Timedelta(days=H))).reindex(s.index).astype(int)
    return s


def design_matrix(s):
    dummies = [pd.get_dummies(s[col], prefix=col, dtype=float).drop(columns=f"{col}_{ref}", errors="ignore")
               for col, ref in REFERENCE.items()]
    return sm.add_constant(pd.concat([s[CONTINUOUS]] + dummies, axis=1), has_constant="add")


train = pd.concat([build_sample(t0).assign(snapshot=t0.date()) for t0 in TRAIN_SNAPSHOTS])
test = build_sample(TEST_T0).assign(snapshot=TEST_T0.date())

summary = pd.concat([train, test.assign(snapshot=f"{TEST_T0.date()} (test)")]).groupby("snapshot", sort=False)["y"]
summary = pd.DataFrame({"merchants": summary.size(), "Y_time = 1": summary.sum(), "rate (%)": (summary.mean() * 100).round(2)})
summary.loc["train total"] = [len(train), train["y"].sum(), round(train["y"].mean() * 100, 2)]
print(f"training rows: {len(train):,} from {train.index.nunique():,} merchants")
summary

training rows: 51,310 from 13,885 merchants


,merchants,Y_time = 1,rate (%)
snapshot,,,
2041-08-01,9822.0,110.0,1.12
2041-09-01,10057.0,108.0,1.07
2041-10-01,10192.0,101.0,0.99
2041-11-01,10475.0,101.0,0.96
2041-12-01,10764.0,100.0,0.93
2042-03-25 (test),11005.0,110.0,1.00
train total,51310.0,520.0,1.01


## 3. Fit on the training snapshots

Training rows are (merchant, snapshot) pairs from all training snapshots, stacked. A merchant can appear in several snapshots, so standard errors are clustered by merchant.

$$\log \frac{P(Y^{\text{time}}_{it} = 1 \mid X_{it})}{1 - P(Y^{\text{time}}_{it} = 1 \mid X_{it})} = \beta_0 + \beta^\top X_{it}$$

fitted by maximum likelihood; $e^{\beta}$ is the odds ratio.

In [6]:
X_train = design_matrix(train)
groups = pd.factorize(train.index)[0]  # cluster by merchant
model = sm.Logit(train["y"], X_train).fit(disp=False, cov_type="cluster", cov_kwds={"groups": groups})
ci = model.conf_int()
pd.DataFrame({
    "coef": model.params,
    "odds ratio": np.exp(model.params),
    "OR 95% CI low": np.exp(ci[0]),
    "OR 95% CI high": np.exp(ci[1]),
    "p-value": model.pvalues,
}).round(3)

,coef,odds ratio,OR 95% CI low,OR 95% CI high,p-value
const,-3.714,0.024,0.010,0.058,0.000
log_volume,0.115,1.121,1.045,1.204,0.002
log_checkout_volume,-0.030,0.970,0.916,1.027,0.298
log_payment_link_volume,0.104,1.109,1.052,1.170,0.000
active_rate,-1.055,0.348,0.186,0.651,0.001
recency,-0.009,0.991,0.985,0.997,0.002
log_tenure,-0.242,0.785,0.700,0.880,0.000
industry_Business services,1.125,3.080,2.159,4.395,0.000
industry_Clothing & accessory,-0.802,0.449,0.175,1.152,0.096
industry_Digital goods,0.715,2.045,1.166,3.588,0.013


## 4. Evaluate on the test snapshot

Lift = adoption rate among the selected merchants ÷ adoption rate in the whole test sample.

In [7]:
test["score"] = model.predict(design_matrix(test).reindex(columns=X_train.columns, fill_value=0.0))
print(f"test AUC: {roc_auc_score(test['y'], test['score']):.3f}, test PR-AUC: {average_precision_score(test['y'], test['score']):.3f} (random: {test['y'].mean():.3f})")

base_rate = test["y"].mean()
by_score = test.sort_values("score", ascending=False)
selections = {
    "random (whole sample)": test,
    "model: top 1%": by_score.head(len(test) // 100),
    f"model: top {TOP_K}": by_score.head(TOP_K),
}
pd.DataFrame({
    name: {"merchants selected": len(sel), "Y_time = 1": int(sel["y"].sum()),
           "rate (%)": round(sel["y"].mean() * 100, 2), "lift": round(sel["y"].mean() / base_rate, 2)}
    for name, sel in selections.items()
}).T

test AUC: 0.727, test PR-AUC: 0.032 (random: 0.010)


,merchants selected,Y_time = 1,rate (%),lift
random (whole sample),11005.0,110.0,1.00,1.00
model: top 1%,110.0,7.0,6.36,6.37
model: top 100,100.0,7.0,7.00,7.00


## 5. Target list

Score the risk set at the day after the last day of data and keep the `TOP_K` highest. Medium/large → sales, small → marketing.

In [8]:
t_score = END + pd.Timedelta(days=1)
pool = build_sample(t_score, with_outcome=False)
pool["score"] = model.predict(design_matrix(pool).reindex(columns=X_train.columns, fill_value=0.0))

targets = (
    pool.sort_values("score", ascending=False).head(TOP_K)
    .assign(rank=lambda t: np.arange(1, len(t) + 1),
            channel=lambda t: np.where(t["business_size"] == "small", "marketing", "sales"))
    [["rank", "score", "channel", "industry", "country", "business_size"] + CONTINUOUS]
)
targets.to_csv(TARGET_LIST_PATH)
print(f"pool: {len(pool):,} merchants; list: top {len(targets)}")
targets.head(10)

pool: 11,402 merchants; list: top 100


,rank,score,channel,industry,country,business_size,log_volume,log_checkout_volume,log_payment_link_volume,active_rate,recency,log_tenure
merchant,,,,,,,,,,,,
84627c91,1,0.175916,marketing,Personal services,Rest of world,small,9.927112,0.0,9.813508,0.159091,1,3.784190
740224c1,2,0.175258,marketing,Business services,US,small,10.457401,0.0,9.575053,0.078125,7,4.158883
2bd7dae3,3,0.156469,marketing,Personal services,US,small,9.680031,0.0,9.680031,0.071429,5,4.025352
ab6e6c9a,4,0.151058,marketing,Personal services,US,small,9.778774,0.0,9.778774,0.250000,3,3.583519
67cdfdbd,5,0.143443,marketing,Business services,Rest of world,small,8.323807,0.0,8.323807,0.258065,1,3.433987
cd9a6461,6,0.142849,marketing,Personal services,Rest of world,small,9.661016,0.0,9.661016,0.055556,22,4.276666
a11c5d49,7,0.142056,marketing,Business services,US,small,10.596972,0.0,10.021870,0.100000,7,5.351858
92e17b8d,8,0.135733,marketing,Business services,US,small,7.958926,0.0,7.601402,0.133333,0,3.401197
fa0c65ad,9,0.127418,marketing,Business services,US,small,7.844241,0.0,7.844241,0.031250,1,4.158883
